# Python 异常处理完全指南

> 目标：从「try/except 语法」到「工程级异常设计」，逐个知识点配合**可运行示例**击破。
> 阅读方式：按顺序执行每个代码单元，观察输出，再对照上面的原理说明。

---

## 学习目标

1. 说清异常的**对象模型**与**继承层次**（为什么分 `BaseException` / `Exception`）
2. 掌握 `try / except / else / finally` 的**完整语义与执行顺序**，并会用代码证明
3. 会设计**自定义异常层次**（带结构化字段，不遮蔽内置异常名）
4. 理解**异常链**（`raise from` / `__cause__` / `__context__`）与**异常组**（`ExceptionGroup` / `except*`）
5. 会用**上下文管理器**与**装饰器**封装异常逻辑（抑制、事务、重试、日志、转换）
6. 掌握 10+ 个实战模式，能识别 14 个常见陷阱的「症状与解法」

## 一句话本质

> **异常不是错误，而是控制流的一部分。好的异常设计让调用者知道「出了什么问题、为什么、该怎么办」，而不是只看到一个 traceback。**

## 业务场景约定

本 Notebook 示例统一取材于 **LLM API 服务开发**（对应项目 `llmops-api`）：
配置加载、调用 LLM 网关、请求参数校验、批量任务、失败降级、用户友好报错。

> 说明：原始需求中的业务场景占位符未填写，此处按项目目录 `LLMOps/llmops-api` 推断为「LLM API 服务开发」。若需换成其他场景，把示例里的名词（网关/模型/配额）替换即可，骨架不变。

In [1]:
# 知识点 0：先确认运行环境支持本 Notebook 用到的全部特性
import sys

print("Python 版本:", sys.version.split()[0])
print("支持异常组 ExceptionGroup / except* :", sys.version_info >= (3, 11))
print("支持异常链 raise ... from ...        :", sys.version_info >= (3, 0))

Python 版本: 3.13.12
支持异常组 ExceptionGroup / except* : True
支持异常链 raise ... from ...        : True


# 前置知识：读懂异常处理需要的 4 块地基

> **本节解决什么问题**：异常处理不是孤立语法——except 的匹配、异常携带信息、finally 的替代品、异常逻辑的封装，分别依赖 4 个语言基础。每个基础都配一个验证示例。

| # | 地基 | 为什么异常处理需要它 |
|---|------|---------------------|
| 1 | 类与继承 | `except X` 匹配的本质是 `issubclass(实际异常, X)` |
| 2 | 一切皆对象 | 异常实例可以携带任意属性（自定义异常的基础） |
| 3 | `with` 上下文管理协议 | `finally` 的工程化替代（`__exit__` 能看到异常） |
| 4 | 装饰器 + `functools.wraps` | 重试 / 日志 / 转换等异常逻辑的标准载体 |

In [2]:
# 前置 1：类继承 —— except 匹配的底层依据就是 issubclass 判断
class Animal:
    pass

class Dog(Animal):
    pass

print(issubclass(Dog, Animal))    # True：Dog 是 Animal 的子类
print(issubclass(Animal, Dog))    # False：父子关系不可反推
print(Dog.__mro__)                # 方法解析顺序：Dog → Animal → object

True
False
(<class '__main__.Dog'>, <class '__main__.Animal'>, <class 'object'>)


In [3]:
# 前置 2：一切皆对象 —— 异常实例同样是一个普通对象
class AppError(Exception):
    pass

e = AppError("出错了")     # 异常实例：先创建，后（才可能被）抛出
e.code = 500               # 像普通对象一样动态绑定属性
print(type(e))             # <class '__main__.AppError'>
print(isinstance(e, Exception))
print(e.args)              # ('出错了',) —— 构造参数自动存入 args

<class '__main__.AppError'>
True
('出错了',)


In [4]:
# 前置 3：with 的本质 —— __enter__ / __exit__ 两个方法
class Demo:
    def __enter__(self):
        print("__enter__：进入 with 块")
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        # 三个参数：异常类型 / 异常实例 / traceback；无异常时全为 None
        print("__exit__：离开 with 块，exc_type =", exc_type)
        return False    # False = 不吞异常；True = 吞掉（第 6 章详讲）

with Demo():
    print("with 块内（无异常）")

try:
    with Demo():
        raise ValueError("块内抛异常")
except ValueError as e:
    print("异常成功穿透 __exit__（返回 False）：", e)

__enter__：进入 with 块
with 块内（无异常）
__exit__：离开 with 块，exc_type = None
__enter__：进入 with 块
__exit__：离开 with 块，exc_type = <class 'ValueError'>
异常成功穿透 __exit__（返回 False）： 块内抛异常


In [5]:
# 前置 4：装饰器 + functools.wraps —— 保持被装饰函数的元信息
from functools import wraps

def trace(func):
    @wraps(func)                     # 没有它，add.__name__ 会变成 "wrapper"
    def wrapper(*args, **kwargs):
        print(f"调用 {func.__name__}，参数 = {args}")
        return func(*args, **kwargs)
    return wrapper

@trace
def add(a, b):
    return a + b

print(add(1, 2))
print("函数名保留:", add.__name__)    # add（而不是 wrapper）

调用 add，参数 = (1, 2)
3
函数名保留: add


# 第 1 章 异常模型

> **本章解决什么问题**：`except` 到底是怎么「匹配」到异常的？为什么 Ctrl+C 拦不住 `except Exception`？答案全在异常的类层次里。

## 1.1 异常也是对象

Python 中一切皆对象，异常也不例外。**所有异常都继承自 `BaseException`**，`except` 捕获时按「是否为其子类」匹配。

## 1.2 异常层次（记忆版）

```text
BaseException
├── SystemExit              # sys.exit() 抛出
├── KeyboardInterrupt       # Ctrl+C
├── GeneratorExit           # 生成器被关闭
└── Exception               # ← 业务代码只需要关心这一支
    ├── ArithmeticError
    │   ├── ZeroDivisionError
    │   ├── OverflowError
    │   └── FloatingPointError
    ├── LookupError
    │   ├── IndexError
    │   └── KeyError
    ├── OSError             # 文件/网络错误的公共基类
    │   ├── FileNotFoundError
    │   ├── PermissionError
    │   ├── TimeoutError
    │   └── ConnectionError
    ├── ValueError / TypeError / AttributeError / NameError
    ├── RuntimeError / StopIteration / StopAsyncIteration
    ├── ImportError
    │   └── ModuleNotFoundError
    ├── AssertionError
    └── NotImplementedError
```

In [6]:
# 知识点 1：用 issubclass 验证异常层次（except 匹配的底层依据）
print(issubclass(ValueError, Exception))             # True
print(issubclass(Exception, BaseException))          # True
print(issubclass(KeyboardInterrupt, BaseException))  # True
print(issubclass(KeyboardInterrupt, Exception))      # False —— 关键！

# OSError 家族：文件 / 网络相关错误的公共基类
print(issubclass(FileNotFoundError, OSError))        # True
print(issubclass(PermissionError, OSError))          # True
print(issubclass(TimeoutError, OSError))             # True（3.10 起与 asyncio 超时统一）
print(issubclass(ConnectionError, OSError))          # True

True
True
True
False
True
True
True
True


## 1.3 为什么分 `BaseException` 和 `Exception`

| 分层 | 直属子类 | 设计意图 | 业务代码应该捕获吗 |
|------|----------|----------|--------------------|
| `BaseException` | `SystemExit`、`KeyboardInterrupt`、`GeneratorExit` | 系统级「退出信号」，不是错误 | **不应该**（捕获后程序无法退出/中断） |
| `Exception` | 绝大多数运行时错误 | 业务异常的总出口 | 可以兜底，但优先捕获具体类型 |

`KeyboardInterrupt` 不被 `except Exception` 捕获是**有意设计**：否则一个 `while True` 循环里写了 `except Exception: pass`，用户就永远无法用 Ctrl+C 杀掉它。

In [7]:
# 知识点 2：Ctrl+C 为什么拦不住 —— KeyboardInterrupt 穿透 except Exception
def interrupted():
    raise KeyboardInterrupt("模拟用户按下 Ctrl+C")

# 内层 except Exception 捕获不到 KeyboardInterrupt → 异常继续向外传播
try:
    try:
        interrupted()
    except Exception:
        print("内层捕获不到（这行不会执行）")
except BaseException as e:
    print("穿透 except Exception，被外层 BaseException 捕获：", type(e).__name__)

# SystemExit（sys.exit() 抛出）同理
try:
    try:
        raise SystemExit(1)
    except Exception:
        print("同样捕获不到")
except BaseException as e:
    print("SystemExit 也穿透了 except Exception：", type(e).__name__)

穿透 except Exception，被外层 BaseException 捕获： KeyboardInterrupt
SystemExit 也穿透了 except Exception： SystemExit


## 1.4 异常对象属性

| 属性 / 用法 | 含义 |
|-------------|------|
| `type(e)` | 异常类 |
| `str(e)` | 给人看的错误消息 |
| `e.args` | 构造异常时传入的参数元组 |
| `e.__traceback__` | traceback 对象（异常传播路径） |
| `e.__cause__ / e.__context__` | 异常链（第 4 章） |

In [8]:
# 知识点 3：异常实例上有什么 —— args / __traceback__
try:
    1 / 0
except ZeroDivisionError as e:
    print("类型        :", type(e).__name__)     # ZeroDivisionError
    print("str(e)      :", e)                    # division by zero
    print("e.args      :", e.args)               # ('division by zero',)
    print("有 traceback:", e.__traceback__ is not None)

# args 可以传多个值（自定义异常常用它携带结构化信息）
err = ValueError("code", 42)
print("多参数 args :", err.args)

类型        : ZeroDivisionError
str(e)      : division by zero
e.args      : ('division by zero',)
有 traceback: True
多参数 args : ('code', 42)


# 第 2 章 try / except / else / finally 完整语法

> **本章解决什么问题**：四个代码块各自何时执行？`else` 存在的意义是什么？`finally` 有哪些会**改变程序语义**的陷阱？

## 2.1 完整骨架与执行顺序

```python
try:
    result = risky()                # ① 先执行
except ValueError as e:             # ② 有异常且类型匹配 → 执行（按顺序取第一个命中的）
    ...
except (TypeError, KeyError) as e:  #    元组 = 多类型任一匹配
    ...
except Exception as e:             #    兜底（放最后）
    ...
else:
    ...                             # ③ 无异常时执行（except 与 else 互斥）
finally:
    ...                             # ④ 无论如何都执行（清理资源）
```

| 块 | 执行时机 | 典型用途 | 陷阱 |
|----|----------|----------|------|
| `try` | 总是 | 只放「可能出错的代码」 | 范围过大 → 误捕获 |
| `except` | 有异常且匹配 | 分类处理 | 裸 `except`、静默吞掉 |
| `else` | 无异常时 | try 的「成功续集」 | 很少人知道它 |
| `finally` | 总是 | 清理资源 | 内部 `return`/`raise` 会覆盖原语义 |

In [9]:
# 知识点 4：用"事件记录"证明四段式的执行顺序
events = []

def risky(ok):
    if not ok:
        raise ValueError("risky 失败")
    return "结果"

# 路径 A：无异常
events.clear()
try:
    events.append("try")
    result = risky(True)
except ValueError:
    events.append("except")
else:
    events.append("else")
finally:
    events.append("finally")
print("无异常路径:", events)

# 路径 B：有异常
events.clear()
try:
    events.append("try")
    result = risky(False)
except ValueError:
    events.append("except")
else:
    events.append("else")
finally:
    events.append("finally")
print("有异常路径:", events)

# 结论：有异常走 except、跳过 else；无异常跳过 except、走 else；finally 永远执行

无异常路径: ['try', 'else', 'finally']
有异常路径: ['try', 'except', 'finally']


## 2.2 `else` 的意义：缩小 try 范围

`else` 只在 try 无异常时执行，**且 else 里的异常不会被前面的 except 捕获**。

| 写法 | `process()` 抛 ValueError 时 | 后果 |
|------|------------------------------|------|
| `process` 写在 try 内 | 流进 `except Exception` | 被**误当成"加载失败"**处理 |
| `process` 写在 else 内 | 不经过 except，直接向外传播 | 调用者看到**真实类型** |

In [10]:
# 知识点 5：else 的价值 —— 后续逻辑的异常不会被"加载错误"的 except 误捕
def load_ok():
    return {"k": 1}

def process(data):
    raise ValueError("处理阶段出错")

# 写法一（不好）：process 混进 try，错误被"加载兜底"误捕
try:
    data = load_ok()
    process(data)                # ← 业务逻辑混进了 try 块
except Exception as e:           # 本意只想兜底"加载"的错
    print("写法一：处理的错误被误捕 →", type(e).__name__)

# 写法二（推荐）：try 只包加载，process 放 else
try:
    try:
        data = load_ok()
    except Exception:
        print("加载失败（不会执行）")
    else:
        process(data)            # 异常不经过上面的 except → 直接向外传播
except ValueError as e:
    print("写法二：错误正常传播 →", type(e).__name__, "-", e)

写法一：处理的错误被误捕 → ValueError
写法二：错误正常传播 → ValueError - 处理阶段出错


## 2.3 `finally` 的两个陷阱

finally 几乎总会执行（即使 try 里 `return` 了、except 里又抛了新异常），但它的**返回值和异常会覆盖 try 的**——这是面试和线上事故的高发区。

In [11]:
# 知识点 6：finally 中的 return 会"吞掉" try 的 return
def f():
    try:
        return 1                 # 本想返回 1
    finally:
        return 2                 # finally 的 return 优先级更高，覆盖了 1

print("f() 返回:", f())          # 2 —— 那个 1 被静默丢弃

# 知识点 7：finally 中抛异常会覆盖原异常
def g():
    try:
        raise ValueError("原始异常")
    finally:
        raise TypeError("finally 中的新异常")

try:
    g()
except TypeError as e:
    print("捕获到的是:", type(e).__name__, "-", e)                # 原始 ValueError 已丢失
    print("原始异常只能从 __context__ 找回:", type(e.__context__).__name__)

# 结论：finally 里只做清理，绝不 return、绝不 raise

f() 返回: 2
捕获到的是: TypeError - finally 中的新异常
原始异常只能从 __context__ 找回: ValueError


In [12]:
# 知识点 8：元组捕获多个异常
for bad in ("abc", ["l", "i", "s", "t"]):
    try:
        int(bad)                          # 字符串 → ValueError；列表 → TypeError
    except (ValueError, TypeError) as e:
        print(f"int({bad!r}) 失败:", type(e).__name__)

# 知识点 9：except 按顺序匹配，第一个命中的生效
try:
    {}["missing"]
except LookupError:               # KeyError 是 LookupError 子类：父类写前面也能命中
    print("LookupError 先命中")
except KeyError:
    print("KeyError 后写，永远不会命中（不可达）")

# 推荐顺序：子类在前精确处理，父类在后兜底
try:
    {}["missing"]
except KeyError:
    print("KeyError 先命中（推荐顺序）")
except LookupError:
    print("LookupError 兜底")

int('abc') 失败: ValueError
int(['l', 'i', 's', 't']) 失败: TypeError
LookupError 先命中
KeyError 先命中（推荐顺序）


In [13]:
# 知识点 10：裸 except vs except Exception —— 差别就在 BaseException 那三个
def f():
    raise SystemExit(1)           # sys.exit() 的本质

# 裸 except：等价于 except BaseException，连系统信号都吞（危险，勿模仿）
try:
    f()
except:
    print("裸 except：连 SystemExit 都被吞了 → sys.exit() 失效")

# except Exception：只捕业务异常，系统级异常放行
try:
    try:
        f()
    except Exception:
        print("这行不会执行")
except SystemExit as e:
    print("except Exception 放行了 SystemExit:", type(e).__name__)

# 记住：裸 except 会让 Ctrl+C 和 sys.exit() 同时失效

裸 except：连 SystemExit 都被吞了 → sys.exit() 失效
except Exception 放行了 SystemExit: SystemExit


# 第 3 章 自定义异常

> **本章解决什么问题**：`raise ValueError("xxx")` 足够吗？——当项目变大，需要一套**异常层次**来表达「哪一层出错、调用方该怎么处理」。

## 3.1 最佳实践（先记结论）

| # | 实践 | 原因 |
|---|------|------|
| 1 | 继承 `Exception`，不要继承 `BaseException` | 不拦截系统信号 |
| 2 | 类名以 `Error` 结尾、见名知义 | `RateLimitError` 比 `E1` 可读 |
| 3 | `super().__init__(...)` 必须调用 | 否则 `str(e)` 为空（反例实验室复现） |
| 4 | 必要时携带结构化字段（status / retry_after） | 让调用方能编程式处理，而不只是读消息 |
| 5 | 建立层次：一个项目根 + 分层子类 | 支持按粒度捕获 |
| 6 | **不要遮蔽内置异常名**（如自定义 `TimeoutError`） | 标准库抛的内置异常会捕不到（反例实验室复现） |

In [14]:
# 知识点 11：最小自定义异常 + 按层次捕获
class AppError(Exception):
    """应用异常基类"""

class ValidationError(AppError):
    """校验错误"""

class NotFoundError(AppError):
    """资源不存在"""

try:
    raise ValidationError("字段不合法")
except AppError as e:             # 捕获基类 = 捕获整个家族
    print("按基类捕获:", type(e).__name__, "-", e)

print(issubclass(ValidationError, AppError))

按基类捕获: ValidationError - 字段不合法
True


In [15]:
# 知识点 12：自定义异常携带结构化字段（LLM API 场景）
class APIStatusError(Exception):
    def __init__(self, status, message, detail=None):
        self.status = status           # HTTP 状态码
        self.message = message         # 错误消息
        self.detail = detail           # 附加信息（如响应体）
        super().__init__(f"[{status}] {message}")   # 保证 str(e) 可读

try:
    raise APIStatusError(429, "触发限流", detail={"retry_after": 3})
except APIStatusError as e:
    print("str(e)   :", e)             # [429] 触发限流
    print("e.status :", e.status)      # 429 —— 结构化字段可直接编程访问
    print("e.detail :", e.detail)

str(e)   : [429] 触发限流
e.status : 429
e.detail : {'retry_after': 3}


In [16]:
# 知识点 13：为 LLM API 服务设计异常层次（本 Notebook 后续综合实战直接复用）
class LLMServiceError(Exception):
    """所有 LLM 服务异常的基类（对外统一出口）"""

class ConfigError(LLMServiceError):
    """配置缺失 / 格式错误"""

class TransportError(LLMServiceError):
    """网络层错误"""
    def __init__(self, message, endpoint=None):
        self.endpoint = endpoint
        super().__init__(message)

class APITimeoutError(TransportError):
    """接口超时（注意命名为 APITimeoutError，避免遮蔽内置 TimeoutError）"""

class RateLimitError(LLMServiceError):
    """限流：可重试"""
    def __init__(self, retry_after=1):
        self.retry_after = retry_after
        super().__init__(f"限流，建议 {retry_after}s 后重试")

# 按需捕获：限流单独处理，网络层统一兜底，其余交给基类
demo_errors = (APITimeoutError("请求超时", endpoint="/v1/chat"),
               RateLimitError(5),
               ConfigError("api_key 缺失"))

for err in demo_errors:
    try:
        raise err
    except RateLimitError as e:
        print("限流处理：等待", e.retry_after, "秒")
    except TransportError as e:
        print("网络层处理：", e, "| endpoint =", e.endpoint)
    except LLMServiceError as e:
        print("兜底处理：", e)

网络层处理： 请求超时 | endpoint = /v1/chat
限流处理：等待 5 秒
兜底处理： api_key 缺失


# 第 4 章 异常链

> **本章解决什么问题**：把底层异常「转译」成业务异常时，如何不丢失根因？——`raise ... from ...`。

## 三个关键属性

| 属性 | 谁设置 | traceback 中的表现 | 用途 |
|------|--------|--------------------|------|
| `__cause__` | `raise B from A` | "The above exception was the **direct cause** of..." | 显式因果 |
| `__context__` | except 块中抛新异常（自动） | "**During handling** of the above exception, another..." | 隐式上下文 |
| `__suppress_context__` | `raise B from None` | 不再显示原异常 | 对外隐藏内部细节 |

In [17]:
# 知识点 14：raise ... from ... 显式异常链 —— 转译时保留根因
import json

class ConfigError(Exception):
    pass

def chain(e):
    """沿 __cause__ 打印异常链：新异常 <- 由它引起"""
    parts = [f"{type(e).__name__}: {e}"]
    while e.__cause__ is not None:
        e = e.__cause__
        parts.append(f"{type(e).__name__}: {e}  <-- 根因")
    print("\n    ".join(parts))

def parse_config(text):
    try:
        return json.loads(text)
    except json.JSONDecodeError as e:        # 底层库的异常
        raise ConfigError("配置解析失败") from e   # 转译成业务异常，但保留根因

try:
    parse_config("{broken json")
except ConfigError as e:
    chain(e)
    print("用户看到的:", e)

ConfigError: 配置解析失败
    JSONDecodeError: Expecting property name enclosed in double quotes: line 1 column 2 (char 1)  <-- 根因
用户看到的: 配置解析失败


In [18]:
# 知识点 15：完整 traceback 里的 "direct cause" —— 亲眼看到因果链
import traceback

try:
    try:
        json.loads("not json")
    except json.JSONDecodeError as e:
        raise ValueError("包装后的业务异常") from e
except ValueError as e:
    # format_exception 输出完整因果链；这里过滤 File/源码行，只看结构
    for line in traceback.format_exception(e):
        if not line.startswith(" "):         # 保留 traceback 头、异常行、因果分隔行
            print(line.rstrip())

Traceback (most recent call last):
json.decoder.JSONDecodeError: Expecting value: line 1 column 1 (char 0)

The above exception was the direct cause of the following exception:
Traceback (most recent call last):
ValueError: 包装后的业务异常


In [19]:
# 知识点 16：隐式链 —— 不写 from，Python 自动记录 __context__
import traceback

try:
    try:
        1 / 0
    except ZeroDivisionError:          # 处理过程中又抛了新异常
        raise ValueError("计算失败")     # 没写 from，但旧异常被记入 __context__
except ValueError as e:
    print("新异常:", type(e).__name__, "-", e)
    print("隐式记录的上文:", type(e.__context__).__name__, "-", e.__context__)
    print("--- traceback 中的标记 ---")
    for line in traceback.format_exception(e):
        if not line.startswith(" ") and "above exception" in line:
            print(line.rstrip())

新异常: ValueError - 计算失败
隐式记录的上文: ZeroDivisionError - division by zero
--- traceback 中的标记 ---

During handling of the above exception, another exception occurred:


In [20]:
# 知识点 17：from None —— 主动切断链条，对用户隐藏内部细节
try:
    try:
        1 / 0
    except ZeroDivisionError:
        raise ValueError("对外只说这个") from None    # 抑制上下文
except ValueError as e:
    print("__context__ 仍有记录吗      :", e.__context__ is not None)
    print("__suppress_context__ 的值   :", e.__suppress_context__)   # True：显示时被抑制

# 知识点 18：三种写法的属性对照实验
def raise_with(way):
    try:
        json.loads("x")
    except json.JSONDecodeError as e:
        if way == "from":
            raise ValueError("A") from e
        elif way == "none":
            raise ValueError("B") from None
        else:
            raise ValueError("C")

print(f"{'写法':<10}{'__cause__':<28}{'__suppress_context__'}")
for way in ("from", "none", "default"):
    try:
        raise_with(way)
    except ValueError as e:
        cause = type(e.__cause__).__name__ if e.__cause__ else "None"
        print(f"{way:<10}{cause:<28}{e.__suppress_context__}")

__context__ 仍有记录吗      : True
__suppress_context__ 的值   : True
写法        __cause__                   __suppress_context__
from      JSONDecodeError             True
none      None                        True
default   None                        False


# 第 5 章 异常组（Python 3.11+）

> **本章解决什么问题**：批量校验 10 个字段、并发跑 50 个任务，失败的可能不止一个——一个异常只能装一条信息。`ExceptionGroup` 让你**一次抛出、一次看全**。

| 能力 | 语法 | 说明 |
|------|------|------|
| 抛出一组 | `raise ExceptionGroup("msg", [e1, e2])` | e1/e2 是异常**实例** |
| 按类型分捕 | `except* ValueError as eg` | 每个 `except*` 是独立"类型槽"，可同时命中多个 |
| 查看成员 | `eg.exceptions` / `eg.message` | 嵌套组拆出的可能是**子组** |
| 兼容性 | 仅 3.11+ | 低版本用「聚合异常」模式替代（第 9 章） |

In [21]:
# 知识点 19：ExceptionGroup —— 批量校验场景（LLM 请求参数）
def validate_request(req):
    """校验 LLM 请求参数：收集所有错误后一次性抛出，而不是挤牙膏"""
    errors = []
    if not req.get("model"):
        errors.append(ValueError("model 不能为空"))
    if not isinstance(req.get("temperature", 0.7), float):
        errors.append(TypeError("temperature 必须是 float"))
    if req.get("max_tokens", 100) <= 0:
        errors.append(ValueError("max_tokens 必须为正整数"))
    if errors:
        raise ExceptionGroup("请求校验失败", errors)
    return "ok"

try:
    validate_request({"model": "", "temperature": "0.7", "max_tokens": -1})
except ExceptionGroup as eg:
    print("组消息:", eg.message)
    print("错误数:", len(eg.exceptions))
    for e in eg.exceptions:
        print("  -", type(e).__name__, ":", e)

组消息: 请求校验失败
错误数: 3
  - ValueError : model 不能为空
  - TypeError : temperature 必须是 float
  - ValueError : max_tokens 必须为正整数


In [22]:
# 知识点 20：except* —— 按"类型槽"分别捕获组内异常
try:
    try:
        raise ExceptionGroup("多个错误", [
            ValueError("值错误"),
            TypeError("类型错误"),
            KeyError("键错误"),
        ])
    except* ValueError as eg:
        print("捕获 ValueError 组:", eg.exceptions)
    except* TypeError as eg:
        print("捕获 TypeError 组:", eg.exceptions)
    # KeyError 没有对应的 except* → 作为"剩余异常组"继续向外传播
except* KeyError as eg:
    print("外层捕获到未处理的 KeyError 组:", eg.exceptions)

捕获 ValueError 组: (ValueError('值错误'),)
捕获 TypeError 组: (TypeError('类型错误'),)
外层捕获到未处理的 KeyError 组: (KeyError('键错误'),)


In [23]:
# 知识点 21：嵌套异常组 —— except* 自动拆包，但拆出的可能是"子组"
group = ExceptionGroup("外层", [
    ValueError("v"),
    ExceptionGroup("内层", [TypeError("t"), KeyError("k")]),
])

try:
    raise group
except* ValueError as eg:
    print("ValueError:", eg.exceptions)
except* TypeError as eg:
    # 注意：从嵌套组拆出的仍是一个子组，需要再取一层 .exceptions
    print("TypeError:", eg.exceptions)
    print("  展开子组:", eg.exceptions[0].exceptions)
except* KeyError as eg:
    print("KeyError  :", eg.exceptions)

ValueError: (ValueError('v'),)
TypeError: (ExceptionGroup('内层', [TypeError('t')]),)
  展开子组: (TypeError('t'),)
KeyError  : (ExceptionGroup('内层', [KeyError('k')]),)


## 5.4 使用场景与并发模式

异常组的典型场景：**批量校验、并发任务错误收集、事务多个失败点、解析多个文件**。

`asyncio` 中的标准姿势（依赖事件循环，Markdown 展示，不在此执行）：

```python
import asyncio

async def main():
    results = await asyncio.gather(task1(), task2(), task3(),
                                   return_exceptions=True)   # 失败不中断，收集为返回值
    errors = [r for r in results if isinstance(r, Exception)]
    if errors:
        raise ExceptionGroup("任务失败", errors)
```

> 注：Notebook 内核运行在事件循环里，`asyncio.run()` 不可直接使用；下面用**线程池**演示等价的「并发 + 错误聚合」模式，输出完全确定。

In [24]:
# 知识点 22：并发任务错误收集 —— 线程池 + ExceptionGroup 聚合失败
from concurrent.futures import ThreadPoolExecutor

def call_llm(i):
    """模拟调用 LLM：第 0、3 个任务失败（确定性失败，输出可复现）"""
    if i in (0, 3):
        raise ConnectionError(f"请求 {i} 连接失败")
    return f"请求 {i} 的结果"

ok, errors = [], []
with ThreadPoolExecutor(max_workers=4) as pool:
    futures = [pool.submit(call_llm, i) for i in range(5)]
    for f in futures:                 # 按提交顺序收集 → 输出顺序确定
        try:
            ok.append(f.result())     # result() 会把线程里的异常重新抛给主线程
        except Exception as e:
            errors.append(e)

try:
    raise ExceptionGroup("批量调用部分失败", errors)
except ExceptionGroup as eg:
    print("成功:", ok)
    print("失败:", [f"{type(e).__name__}: {e}" for e in eg.exceptions])

成功: ['请求 1 的结果', '请求 2 的结果', '请求 4 的结果']
失败: ['ConnectionError: 请求 0 连接失败', 'ConnectionError: 请求 3 连接失败']


# 第 6 章 异常与上下文管理器

> **本章解决什么问题**：`finally` 只能"清理"，看不到异常细节；`__exit__` 能拿到异常三元组 `(exc_type, exc_val, exc_tb)` 并决定**吞不吞**——这是抑制、事务、转换的统一基础。

## `__exit__` 的三个参数与返回值

| 项 | 无异常时 | 有异常时 |
|----|----------|----------|
| `exc_type` | `None` | 异常类（如 `ValueError`） |
| `exc_val` | `None` | 异常实例 |
| `exc_tb` | `None` | traceback 对象 |
| **返回值** | 无意义 | `True` = 吞掉异常；`False`/`None` = 继续传播 |

In [25]:
# 知识点 23：__exit__ 返回 True = 吞掉异常（contextlib.suppress 的原理）
class Suppress:
    def __init__(self, *exceptions):
        self.exceptions = exceptions

    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        if exc_type is not None and issubclass(exc_type, self.exceptions):
            print(f"  __exit__ 抑制了 {exc_type.__name__}")
            return True           # True：吞掉名单内的异常
        return False              # False：其余异常照常抛出

with Suppress(ValueError):
    int("abc")                    # 抛 ValueError → 被抑制
print("继续执行（ValueError 被吞掉）")

try:
    with Suppress(ValueError):
        1 / 0                     # ZeroDivisionError 不在名单 → 照常抛出
except ZeroDivisionError as e:
    print("ZeroDivisionError 不受影响:", e)

  __exit__ 抑制了 ValueError
继续执行（ValueError 被吞掉）
ZeroDivisionError 不受影响: division by zero


In [26]:
# 知识点 24：标准库现成的 suppress —— 一行搞定"预期内的失败"
import os
from contextlib import suppress

with suppress(FileNotFoundError):
    os.remove("不存在的文件.txt")          # 删除不存在的文件：不报错
print("suppress(FileNotFoundError)：文件不存在也不报错")

with suppress(KeyError):
    {}["missing"]                          # 取缺失键：不报错
print("suppress(KeyError)：取缺失键也不报错")

suppress(FileNotFoundError)：文件不存在也不报错
suppress(KeyError)：取缺失键也不报错


In [27]:
# 知识点 25：@contextmanager 实现事务 —— 成功提交、失败回滚
from contextlib import contextmanager

class FakeConn:
    """模拟数据库 / 网关连接：用列表记录操作日志（输出确定）"""
    def __init__(self):
        self.log = []
    def begin(self):
        self.log.append("BEGIN")
    def execute(self, sql):
        self.log.append(f"EXEC {sql}")
        if "FAIL" in sql:
            raise RuntimeError("SQL 执行失败")
    def commit(self):
        self.log.append("COMMIT")
    def rollback(self):
        self.log.append("ROLLBACK")

@contextmanager
def transaction(conn):
    conn.begin()
    try:
        yield conn
    except Exception:
        conn.rollback()
        raise                     # 回滚后必须重新抛出，不能吞
    else:
        conn.commit()

# 路径一：成功 → 提交
conn = FakeConn()
with transaction(conn) as c:
    c.execute("INSERT A")
print("成功路径:", conn.log)

# 路径二：失败 → 回滚 + 异常继续传播
conn = FakeConn()
try:
    with transaction(conn) as c:
        c.execute("INSERT FAIL")
except RuntimeError as e:
    print("失败路径:", conn.log, "| 异常:", e)

成功路径: ['BEGIN', 'EXEC INSERT A', 'COMMIT']
失败路径: ['BEGIN', 'EXEC INSERT FAIL', 'ROLLBACK'] | 异常: SQL 执行失败


In [28]:
# 知识点 26：上下文管理器做异常转换（底层错误 → 业务错误）
from contextlib import contextmanager

@contextmanager
def translate_exceptions():
    try:
        yield
    except KeyError as e:
        raise ValueError(f"缺少字段: {e}") from e

try:
    with translate_exceptions():
        {"model": "gpt"}["temperature"]     # KeyError → ValueError
except ValueError as e:
    print("转换后的异常:", type(e).__name__, "-", e)
    print("根因保留:", type(e.__cause__).__name__)

转换后的异常: ValueError - 缺少字段: 'temperature'
根因保留: KeyError


# 第 7 章 异常与装饰器

> **本章解决什么问题**：重试、错误日志、异常转译——这些横切逻辑不该散落在每个函数里，装饰器是标准载体。

| 装饰器 | 职责 | 关键点 |
|--------|------|--------|
| `@retry` | 失败自动重试 | 上限 + 只重试可重试的异常 |
| `@log_exceptions` | 记录 traceback 后重新抛出 | `logger.exception` 自动带栈 |
| `@translate` | 底层异常 → 业务异常 | `from e` 保留根因 |

In [29]:
# 知识点 27：重试装饰器 —— 网络抖动的标准解法
import time
from functools import wraps

def retry(times=3, delay=0, exceptions=(Exception,)):
    """times：最大尝试次数；exceptions：只重试这些异常"""
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            for i in range(1, times + 1):
                try:
                    return func(*args, **kwargs)
                except exceptions as e:
                    if i == times:
                        raise                      # 最后一次失败：原样抛出
                    print(f"  第 {i} 次失败: {e}，准备重试")
                    time.sleep(delay)              # 教学示例 delay=0；真实场景用指数退避
        return wrapper
    return decorator

_ATTEMPTS = {"n": 0}

@retry(times=3, delay=0, exceptions=(ConnectionError,))
def call_llm():
    """模拟 LLM 接口：前 2 次失败，第 3 次成功（确定性，输出可复现）"""
    _ATTEMPTS["n"] += 1
    if _ATTEMPTS["n"] <= 2:
        raise ConnectionError("网关超时")
    return "LLM 响应内容"

print("最终结果:", call_llm())

  第 1 次失败: 网关超时，准备重试
  第 2 次失败: 网关超时，准备重试
最终结果: LLM 响应内容


In [30]:
# 知识点 28：异常日志装饰器 —— logger.exception 自动带 traceback
import io
import logging
from functools import wraps

# 教学环境：日志写入内存流，便于嵌入 Notebook 输出（真实项目写文件 / stdout）
_LOG_BUF = io.StringIO()
logging.basicConfig(handlers=[logging.StreamHandler(_LOG_BUF)],
                    format="%(levelname)s %(name)s: %(message)s",
                    level=logging.INFO, force=True)
logger = logging.getLogger("llmops")

def flush_log():
    """把累积的日志打到屏幕并清空（本 Notebook 后续单元也会用到）"""
    text = _LOG_BUF.getvalue()
    _LOG_BUF.truncate(0)
    _LOG_BUF.seek(0)
    print(text, end="")

def log_exceptions(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        try:
            return func(*args, **kwargs)
        except Exception:
            logger.exception(f"{func.__name__} 执行失败")   # error 级 + 完整 traceback
            raise                                            # 记录完再抛出，绝不吞
    return wrapper

@log_exceptions
def embed(texts):
    return {}["missing_key"]        # 模拟内部 KeyError

try:
    embed(["你好"])
except KeyError as e:
    print("调用者仍然拿到原始异常:", type(e).__name__)
print("--- 日志内容（含 traceback）---")
flush_log()

调用者仍然拿到原始异常: KeyError
--- 日志内容（含 traceback）---
ERROR llmops: embed 执行失败
Traceback (most recent call last):
  File "<cell 43>", line 24, in wrapper
  File "<cell 43>", line 32, in embed
KeyError: 'missing_key'


In [31]:
# 知识点 29：异常转换装饰器 —— 边界层统一翻译错误
from functools import wraps

def translate(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        try:
            return func(*args, **kwargs)
        except KeyError as e:
            raise ValueError(f"缺少键: {e}") from e
    return wrapper

@translate
def get_model(config):
    return config["model"]

try:
    get_model({"temperature": 0.7})
except ValueError as e:
    print("对外异常:", e)
    print("根因:", type(e.__cause__).__name__)

对外异常: 缺少键: 'model'
根因: KeyError


# 第 8 章 断言与异常

> **本章解决什么问题**：`assert` 和 `raise` 都是抛异常，边界在哪？——一句话：**assert 管「不可能」，raise 管「不该」**。

| 对比项 | `assert` | `if + raise` |
|--------|----------|--------------|
| 抛出 | `AssertionError` | 任意异常 |
| `python -O` 下 | **被整体移除** | 照常执行 |
| 适用 | 内部不变量、调试检查、测试 | 参数校验、业务规则 |

In [32]:
# 知识点 30：assert 的语义与"等价写法"
def divide(a, b):
    assert b != 0, "除数不能为 0"       # 语法糖
    return a / b

print("正常:", divide(10, 2))

try:
    divide(1, 0)
except AssertionError as e:            # assert 失败抛 AssertionError
    print("断言失败:", e)

# 等价于：
# if not (b != 0):
#     raise AssertionError("除数不能为 0")

正常: 5.0
断言失败: 除数不能为 0


In [33]:
# 知识点 31：python -O 会移除所有 assert —— 用子进程实测证明
import subprocess
import sys

snippet = "assert 1 == 2, '断言失败'\nprint('assert 被跳过，继续执行')"

for args, label in [([], "普通模式"), (["-O"], "-O 优化模式")]:
    r = subprocess.run([sys.executable, *args, "-c", snippet],
                       capture_output=True, text=True)
    print(f"{label}: 退出码 = {r.returncode}")
    print(f"  stdout      = {r.stdout.strip()!r}")
    print(f"  stderr 末行 = {r.stderr.strip().splitlines()[-1] if r.stderr else '(无)'}")

# 结论：-O 模式下断言被移除、程序"正常"跑完 —— 所以参数校验绝不能依赖 assert

普通模式: 退出码 = 1
  stdout      = ''
  stderr 末行 = AssertionError: 断言失败
-O 优化模式: 退出码 = 0
  stdout      = 'assert 被跳过，继续执行'
  stderr 末行 = (无)


In [34]:
# 知识点 32：assert 只用于"内部不变量"，参数校验必须 if + raise
# 反例（勿模仿）：python -O 后这行校验会消失
def withdraw_bad(amount):
    assert amount > 0, "金额必须为正"
    return f"扣款 {amount}"

# 正例：运行时校验，任何模式下都生效
def withdraw(amount):
    if amount <= 0:
        raise ValueError("金额必须为正")
    return f"扣款 {amount}"

try:
    withdraw(-100)
except ValueError as e:
    print("参数校验生效:", e)

# assert 的正确用途：检查"不应该发生"的内部状态（如循环不变量）
def binary_search(arr, target):
    lo, hi = 0, len(arr) - 1
    while lo <= hi:
        mid = (lo + hi) // 2
        assert 0 <= mid < len(arr), "循环不变量被破坏"
        if arr[mid] == target:
            return mid
        elif arr[mid] < target:
            lo = mid + 1
        else:
            hi = mid - 1
    return -1

print("二分查找结果:", binary_search([1, 3, 5, 7, 9], 7))

参数校验生效: 金额必须为正
二分查找结果: 3


# 第 9 章 异常设计模式

> **本章解决什么问题**：单点的 try/except 会写了，但系统级错误策略怎么定？——6 种模式按场景取用。

| 模式 | 一句话 | 适用场景 |
|------|--------|----------|
| 快速失败 | 启动时立刻报错 | 配置 / 依赖检查 |
| 错误累积 | 攒齐一起报 | 表单/参数校验 |
| 异常转换 | 底层 → 业务 | 分层架构的边界 |
| 重试+退避 | 失败后按指数间隔重试 | 网络/网关抖动 |
| 断路器 | 连续失败后熔断快速失败 | 保护下游 |
| Result 模式 | 用返回值表达可预期失败 | 不想用异常控制流程时 |

In [35]:
# 知识点 33：快速失败（Fail Fast）—— 越早报错，定位成本越低
import os

def parse_config(path):
    if not os.path.exists(path):          # 启动时立刻检查，而不是半夜调用时才炸
        raise FileNotFoundError(f"配置不存在: {path}")
    with open(path, encoding="utf-8") as f:
        return f.read()

try:
    parse_config("/tmp/不存在的配置.yaml")     # 教学演示：假路径
except FileNotFoundError as e:
    print("启动阶段就暴露问题:", e)

启动阶段就暴露问题: 配置不存在: /tmp/不存在的配置.yaml


In [36]:
# 知识点 34：错误累积 —— 一次汇报所有问题，而不是挤牙膏
class ValidationErrors(Exception):
    def __init__(self, errors):
        self.errors = errors
        super().__init__(f"共 {len(errors)} 个错误")

def validate(data):
    errors = []
    if "name" not in data:
        errors.append("缺少 name")
    if "age" not in data:
        errors.append("缺少 age")
    if errors:
        raise ValidationErrors(errors)      # 攒够一起抛

try:
    validate({})
except ValidationErrors as e:
    print(e)
    print("明细:", e.errors)

共 2 个错误
明细: ['缺少 name', '缺少 age']


In [37]:
# 知识点 35：指数退避重试 —— 固定随机种子 + 替换 sleep，保证演示可复现
import random
import time

random.seed(42)                       # 固定种子：每次运行退避时长完全一致

_sleep_log = []
def fake_sleep(sec):                  # 替换 time.sleep：不真等待，只记录
    _sleep_log.append(round(sec, 2))

real_sleep, time.sleep = time.sleep, fake_sleep

_ATTEMPT = {"n": 0}
def flaky():
    """前 3 次失败，第 4 次成功"""
    _ATTEMPT["n"] += 1
    if _ATTEMPT["n"] <= 3:
        raise ConnectionError("抖动")
    return "ok"

def retry_with_backoff(func, max_retries=5):
    for i in range(max_retries):
        try:
            return func()
        except Exception:
            if i == max_retries - 1:
                raise
            delay = min(2 ** i + random.random(), 30)   # 指数退避 + 随机抖动
            time.sleep(delay)                            # 已替换为记录
    return None

print("结果:", retry_with_backoff(flaky))
print("各次重试的退避时长(秒):", _sleep_log)
time.sleep = real_sleep                # 恢复，避免影响后续单元

结果: ok
各次重试的退避时长(秒): [1.64, 2.03, 4.28]


In [38]:
# 知识点 36：断路器 —— 连续失败达阈值后"熔断"，直接快速失败
class CircuitBreaker:
    def __init__(self, threshold=3, timeout=60, clock=None):
        self.threshold = threshold        # 连续失败阈值
        self.timeout = timeout            # 熔断多久后半开试探
        self.failures = 0
        self.last_failure = 0
        self.open = False
        self._clock = clock or time.time  # 注入时钟：教学环境完全可复现

    def call(self, func, *args, **kwargs):
        if self.open:
            if self._clock() - self.last_failure > self.timeout:
                self.open = False         # 半开：超时后放行试探
                self.failures = 0
            else:
                raise RuntimeError("断路器打开，快速失败")
        try:
            result = func(*args, **kwargs)
            self.failures = 0             # 成功即复位
            return result
        except Exception:
            self.failures += 1
            self.last_failure = self._clock()
            if self.failures >= self.threshold:
                self.open = True          # 达到阈值：熔断
            raise

# 用"假时钟"驱动：确定性演示熔断 → 半开 → 恢复
NOW = [1000.0]
def fake_clock():
    return NOW[0]

def always_fail():
    raise ConnectionError("下游挂了")

cb = CircuitBreaker(threshold=3, timeout=60, clock=fake_clock)

for i in range(3):
    try:
        cb.call(always_fail)
    except ConnectionError as e:
        print(f"第 {i+1} 次调用失败: {e}（open={cb.open}）")

try:
    cb.call(always_fail)                 # 熔断后不再真正调用下游
except RuntimeError as e:
    print("熔断生效:", e)

NOW[0] += 61                              # 时间前进 61 秒：超过 timeout，半开恢复
def healthy():
    return "下游恢复了"

print("超时后放行试探:", cb.call(healthy), "（failures 已复位 =", cb.failures, "）")

第 1 次调用失败: 下游挂了（open=False）
第 2 次调用失败: 下游挂了（open=False）
第 3 次调用失败: 下游挂了（open=True）
熔断生效: 断路器打开，快速失败
超时后放行试探: 下游恢复了 （failures 已复位 = 0 ）


## 9.6 Result 模式

> **修正说明**：源文档写作 `value: any`——小写 `any` 是内置函数不是类型，作为注解虽能运行但语义错误。本 Notebook 修正为 `typing.Any`。

用返回值代替异常表达「**可预期的失败**」（如除零、找不到）：调用方用 `match` 消费，调用栈不被打断。

In [39]:
# 知识点 37：Result 模式 —— 可预期的失败用返回值，而不是异常
# 【修正说明】源文档的 `value: any` 修正为 typing.Any
from dataclasses import dataclass
from typing import Any

@dataclass
class Ok:
    value: Any

@dataclass
class Err:
    error: Exception

def divide(a, b):
    if b == 0:
        return Err(ZeroDivisionError("除零"))     # 可预期的失败：返回而不是抛
    return Ok(a / b)

for a, b in [(10, 2), (1, 0)]:
    result = divide(a, b)
    match result:
        case Ok(value):
            print(f"{a}/{b} = {value}")
        case Err(error):
            print(f"{a}/{b} 失败: {error}")

10/2 = 5.0
1/0 失败: 除零


# 第 10 章 实战场景（LLM API 服务）

> **本章解决什么问题**：把前面零散的语法落到 `llmops-api` 的真实环节——接口调用、配置读取、批量任务、降级、用户报错。

In [40]:
# 知识点 38：API 错误处理 —— 把 HTTP 状态码翻译成异常层次
class APIError(Exception):
    def __init__(self, status, message, detail=None):
        self.status = status
        self.message = message
        self.detail = detail
        super().__init__(f"[{status}] {message}")

class FakeResponse:
    """模拟 requests 的 Response（确定性）"""
    def __init__(self, status_code, payload=None, text=""):
        self.status_code = status_code
        self._payload = payload
        self.text = text
    def json(self):
        return self._payload
    def raise_for_status(self):
        if self.status_code >= 400:
            raise APIError(self.status_code, "HTTP 错误", self.text)

def handle_response(resp):
    if resp.status_code == 404:
        raise APIError(404, "资源不存在")
    if resp.status_code == 500:
        raise APIError(500, "服务器错误", resp.text)   # 附上响应体便于排查
    resp.raise_for_status()                  # 其余 4xx/5xx 统一兜底
    return resp.json()

for resp in (FakeResponse(404),
             FakeResponse(500, text="upstream boom"),
             FakeResponse(429),
             FakeResponse(200, payload={"content": "你好"})):
    try:
        print("成功:", handle_response(resp))
    except APIError as e:
        print(f"APIError: status={e.status}, detail={e.detail}")

APIError: status=404, detail=None
APIError: status=500, detail=upstream boom
APIError: status=429, detail=
成功: {'content': '你好'}


In [41]:
# 知识点 39：配置文件读取 —— 缺文件 / 格式错误分别转译（教学临时文件用完即删）
import json
import os

class ConfigError(Exception):
    pass

CONFIG_PATH = "_study_demo_config.json"

def read_config(path):
    try:
        with open(path, encoding="utf-8") as f:
            return json.load(f)
    except FileNotFoundError:
        raise ConfigError(f"配置文件不存在: {os.path.basename(path)}")
    except json.JSONDecodeError as e:
        raise ConfigError(f"配置格式错误: {e}") from e    # 链保留根因

# 场景一：文件不存在
try:
    read_config(CONFIG_PATH)
except ConfigError as e:
    print("场景一 →", e)

# 场景二：内容非法 JSON
with open(CONFIG_PATH, "w", encoding="utf-8") as f:
    f.write("{ 不是合法 json")
try:
    read_config(CONFIG_PATH)
except ConfigError as e:
    print("场景二 →", e, "| 根因:", type(e.__cause__).__name__)

# 场景三：正常
with open(CONFIG_PATH, "w", encoding="utf-8") as f:
    json.dump({"model": "gpt-4o-mini", "temperature": 0.7}, f, ensure_ascii=False)
print("场景三 →", read_config(CONFIG_PATH))

os.remove(CONFIG_PATH)                     # 清理教学临时文件

场景一 → 配置文件不存在: _study_demo_config.json
场景二 → 配置格式错误: Expecting property name enclosed in double quotes: line 1 column 3 (char 2) | 根因: JSONDecodeError
场景三 → {'model': 'gpt-4o-mini', 'temperature': 0.7}


In [42]:
# 知识点 40：优雅降级 —— API 失败时回退缓存（全程留痕）
_CACHE = {"user_42": {"name": "黄桃", "plan": "pro"}}

def api_get_user(uid):
    raise APIError(503, "服务暂不可用")     # 模拟 API 挂了

def get_user(uid):
    try:
        return api_get_user(uid)
    except APIError:
        logger.warning("API 失败，回退缓存: uid=%s", uid)   # 降级必须留痕
        if uid in _CACHE:
            return _CACHE[uid]
        raise                              # 缓存也没有：把问题抛还给调用者

print("降级命中:", get_user("user_42"))
try:
    get_user("user_missing")               # 缓存也没有 → 不硬扛
except APIError as e:
    print("缓存也没有 →", e)
flush_log()

降级命中: {'name': '黄桃', 'plan': 'pro'}
缓存也没有 → [503] 服务暂不可用
WARNING llmops: API 失败，回退缓存: uid=user_42
WARNING llmops: API 失败，回退缓存: uid=user_missing


In [43]:
# 知识点 41：用户可见错误 vs 内部错误 —— 两类异常分开处理
class UserFacingError(Exception):
    """可以放心展示给最终用户的错误"""

def internal_process(data):
    if "query" not in data:
        raise UserFacingError("请填写查询内容后再试")
    return {}["内部缓存键"]                 # 模拟一个内部 bug（KeyError）

def process(data):
    try:
        return {"result": internal_process(data)}
    except UserFacingError as e:
        return {"error": str(e)}            # 用户错误：原文展示
    except Exception:
        logger.exception("内部错误")         # 内部错误：记日志，绝不外泄细节
        return {"error": "系统错误，请稍后重试"}

print(process({}))                          # 缺 query → 用户可读的错误
print(process({"query": "你好"}))            # 内部 bug → 兜底话术 + 日志留痕
flush_log()

{'error': '请填写查询内容后再试'}
{'error': '系统错误，请稍后重试'}
ERROR llmops: 内部错误
Traceback (most recent call last):
  File "<cell 60>", line 12, in process
  File "<cell 60>", line 8, in internal_process
KeyError: '内部缓存键'


# 反例实验室：错误写法 → 实际报错 → 正确写法

> **本节解决什么问题**：看 100 遍「不要这样做」，不如亲手触发一次。每个反例都**真实运行**并展示症状，再给出正确写法对比。

| # | 反例 | 症状 | 正确写法 |
|---|------|------|----------|
| 1 | 静默吞异常 | 调用方拿到 `None`，毫无感知 | 记日志 + `raise` |
| 2 | 自定义异常忘 `super().__init__` | `str(e)` 为空 | 调用 `super().__init__(...)` |
| 3 | `raise e` 代替裸 `raise` | traceback 多一层重复帧 | 直接 `raise` |
| 4 | 用异常控制正常流程 | 慢、难读 | `.get()` / `in` / `defaultdict` |
| 5 | 循环里 `except: continue` | 批量任务失败被无声跳过 | 收集错误统一汇报 |
| 6 | 自定义类遮蔽内置异常名 | 标准库抛的内置异常捕不到 | 加业务前缀命名 |

> `finally` 中 return/raise、裸 `except`、`assert` 被优化等陷阱，已在第 2、8 章用运行输出复现，此处不再重复。

In [44]:
# 反例 1：静默吞异常 —— 最难排查的坑
def bad():
    try:
        {}["key"]
    except Exception:
        pass                 # ← 出错后无声无息

result = bad()
print("反例：调用者拿到的返回值 =", result, "（完全不知道出过事）")

def good():
    try:
        {}["key"]
    except Exception:
        logger.exception("good() 出错")        # 至少留下现场
        raise                                  # 不掩盖
try:
    good()
except KeyError as e:
    print("正确：调用者能感知失败:", type(e).__name__)
flush_log()

反例：调用者拿到的返回值 = None （完全不知道出过事）
正确：调用者能感知失败: KeyError
ERROR llmops: good() 出错
Traceback (most recent call last):
  File "<cell 62>", line 13, in good
KeyError: 'key'


In [45]:
# 反例 2：自定义异常忘记 super().__init__ —— str(e) 变成空字符串
class BrokenError(Exception):
    def __init__(self, msg):
        self.msg = msg
        # 忘了 super().__init__(msg)   ← 事故现场

class FixedError(Exception):
    def __init__(self, msg):
        self.msg = msg
        super().__init__(msg)               # 正确

for cls in (BrokenError, FixedError):
    e = cls("出错了")
    print(f"{cls.__name__:11}: str(e) = {str(e)!r}, e.args = {e.args}")

BrokenError: str(e) = '出错了', e.args = ('出错了',)
FixedError : str(e) = '出错了', e.args = ('出错了',)


In [46]:
# 反例 3 & 修正说明：except 中 raise e vs 裸 raise
# 源文档称 `raise e` 会"丢失原始 traceback"——这在 Python 2 时代成立；
# Python 3 中两者都保留原始栈，差别是 `raise e` 会追加一个重复的当前帧。
# 用 traceback 帧数实测证明：

def inner():
    1 / 0                                  # 原始出错点

def tb_depth(e):
    """统计 traceback 帧数"""
    n, tb = 0, e.__traceback__
    while tb is not None:
        n += 1
        tb = tb.tb_next
    return n

def way_raise_e():
    try:
        inner()
    except ZeroDivisionError as e:
        raise e                             # 写法一：raise e（不推荐）

def way_bare_raise():
    try:
        inner()
    except ZeroDivisionError:
        raise                               # 写法二：裸 raise（推荐）

for fn in (way_raise_e, way_bare_raise):
    try:
        fn()
    except ZeroDivisionError as e:
        print(f"{fn.__name__:16} traceback 帧数 = {tb_depth(e)}（原始栈都在，但前者多 1 层重复帧）")

way_raise_e      traceback 帧数 = 4（原始栈都在，但前者多 1 层重复帧）
way_bare_raise   traceback 帧数 = 3（原始栈都在，但前者多 1 层重复帧）


In [47]:
# 反例 4：用异常做正常流程控制 —— 慢且难读
d = {"model": "gpt-4o-mini"}

def bad_get(key):
    try:
        return d[key]                       # 用异常处理"键可能不存在"
    except KeyError:
        return None

def good_get(key):
    return d.get(key)                       # 语言为"预期内缺失"提供的无异常写法

print("取缺失键:", bad_get("temperature"), good_get("temperature"))
print("取存在键:", bad_get("model"), good_get("model"))

# 性能提示：异常的抛出/捕获开销远大于一次字典查找（定性结论；
# 计时数字随机器波动，为保证输出可复现此处不做计时演示）。
# 高频路径上"预期中的缺失"应使用 .get() / defaultdict / in 检查。

取缺失键: None None
取存在键: gpt-4o-mini gpt-4o-mini


In [48]:
# 反例 5：循环里 except: continue —— 10 个任务错了 5 个，一个字都不说
def process(item):
    if item % 2 == 1:
        raise ValueError(f"item {item} 处理失败")
    return item * 10

done = 0
for item in range(10):
    try:
        process(item)
        done += 1
    except Exception:
        continue                            # ← 失败被无声跳过
print(f"反例：完成 {done}/10，失败了几个？没人知道")

results, errors = [], []
for item in range(10):
    try:
        results.append(process(item))
    except ValueError as e:
        errors.append(str(e))
print(f"正确：成功 {len(results)} 个，失败 {len(errors)} 个 → {errors[:2]} ...")

反例：完成 5/10，失败了几个？没人知道
正确：成功 5 个，失败 5 个 → ['item 1 处理失败', 'item 3 处理失败'] ...


In [49]:
# 反例 6：自定义异常遮蔽内置名 —— TimeoutError / ConnectionError 都是内置异常
# 一旦写了 `class TimeoutError(Exception)`，本文件后续的 TimeoutError 全指向你的类，
# 标准库抛出的超时将不再被 except TimeoutError 捕获。

print("内置 TimeoutError 是 OSError 子类:", issubclass(TimeoutError, OSError))

class APITimeoutError(Exception):           # 正确：加业务前缀，不遮蔽内置名
    pass

print("两个类互不影响:", APITimeoutError is not TimeoutError)

内置 TimeoutError 是 OSError 子类: True
两个类互不影响: True


# 实用示例集：5 个可直接抄走的工程模板

> **本节解决什么问题**：前面按知识点拆开讲，这里合并成完整模板——每个都围绕 LLM API 开发，含适用场景与注意点。

| # | 模板 | 适用场景 | 主要注意点 |
|---|------|----------|------------|
| 1 | 配置加载 | 进程启动 | 快速失败 + 异常链 |
| 2 | 指数退避重试 | 网关抖动 | 只重试可重试异常 |
| 3 | 批量校验 + 异常组 | 表单/请求校验 | 3.11+ 才有 except* |
| 4 | 优雅降级 | 主服务不可用 | 降级必须留痕 |
| 5 | 事务上下文 | 多步写入 | 回滚后记得 raise |

In [50]:
# 模板 1：配置加载（快速失败 + 异常链 + 统一出口）
import json
import os

class ConfigError(Exception):
    """配置相关错误的统一出口"""

def load_config(path):
    if not os.path.exists(path):
        # ① 快速失败：启动时立刻暴露，不带病运行
        raise ConfigError(f"配置缺失: {path}")
    try:
        with open(path, encoding="utf-8") as f:
            cfg = json.load(f)
    except json.JSONDecodeError as e:
        # ② 异常链：保留根因，方便排查
        raise ConfigError(f"配置格式错误: {path}") from e
    # ③ 业务校验：内容合法才算加载成功
    if "api_key" not in cfg:
        raise ConfigError("缺少必填项 api_key")
    return cfg

CFG = "_tpl_config.json"
with open(CFG, "w", encoding="utf-8") as f:
    json.dump({"model": "gpt-4o-mini"}, f, ensure_ascii=False)

try:
    load_config(CFG)                    # 缺 api_key → 第 3 道防线拦截
except ConfigError as e:
    print("启动被拦截:", e)
os.remove(CFG)

启动被拦截: 缺少必填项 api_key


In [51]:
# 模板 2：指数退避重试装饰器（重试 + 退避 + 日志，三合一）
from functools import wraps

logger = logging.getLogger("llmops.retry")

def retry_backoff(times=4, base=0.1, cap=30, exceptions=(ConnectionError, TimeoutError)):
    """times: 最大尝试次数；base: 初始退避秒数；cap: 退避上限"""
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            for attempt in range(1, times + 1):
                try:
                    return func(*args, **kwargs)
                except exceptions as e:
                    if attempt == times:
                        logger.error("%s 重试 %d 次后仍失败", func.__name__, times)
                        raise
                    # 教学环境不真 sleep，只打印将等待的时长（真实项目 time.sleep）
                    wait = min(base * 2 ** (attempt - 1), cap)
                    print(f"  {func.__name__} 第 {attempt} 次失败({e})，退避 {wait:.1f}s")
        return wrapper
    return decorator

_STATE = {"n": 0}

@retry_backoff(times=4)
def call_gateway():
    _STATE["n"] += 1
    if _STATE["n"] <= 2:
        raise ConnectionError("网关 502")
    return "成功响应"

print("最终结果:", call_gateway())

  call_gateway 第 1 次失败(网关 502)，退避 0.1s
  call_gateway 第 2 次失败(网关 502)，退避 0.2s
最终结果: 成功响应


In [52]:
# 模板 3：批量校验 + ExceptionGroup（一次报出全部问题）
def validate_payload(payload):
    """不通过则抛 ExceptionGroup，内含全部校验错误"""
    errors = []
    if not payload.get("model"):
        errors.append(ValueError("model 不能为空"))
    t = payload.get("temperature", 0.7)
    if not isinstance(t, (int, float)) or not (0 <= t <= 2):
        errors.append(ValueError(f"temperature 非法: {t!r}"))
    if payload.get("max_tokens", 0) <= 0:
        errors.append(ValueError("max_tokens 必须为正"))
    if errors:
        raise ExceptionGroup("请求校验失败", errors)

try:
    validate_payload({"model": "", "temperature": 5, "max_tokens": -1})
except ExceptionGroup as eg:
    print(eg.message)
    for e in eg.exceptions:
        print("  -", e)

请求校验失败
  - model 不能为空
  - temperature 非法: 5
  - max_tokens 必须为正


In [53]:
# 模板 4：优雅降级（主路径失败 → 备用路径，全程留痕）
_CACHE = {}

def with_fallback(primary, fallback, *args):
    """primary 抛异常时尝试 fallback；两者都失败才向上抛"""
    try:
        return primary(*args)
    except Exception as e:
        logger.warning("主路径失败(%s: %s)，尝试降级", type(e).__name__, e)
        try:
            return fallback(*args)
        except Exception:
            logger.error("降级路径也失败，异常继续上抛")
            raise

def hot_path(uid):
    raise TimeoutError("主服务超时")

def cold_cache(uid):
    if uid in _CACHE:
        return _CACHE[uid]
    raise KeyError(uid)

_CACHE["u1"] = {"name": "黄桃"}

print("降级命中:", with_fallback(hot_path, cold_cache, "u1"))
try:
    with_fallback(hot_path, cold_cache, "u9")
except Exception as e:
    print("彻底失败:", type(e).__name__, "-", e)
flush_log()

降级命中: {'name': '黄桃'}
彻底失败: KeyError - 'u9'
WARNING llmops.retry: 主路径失败(TimeoutError: 主服务超时)，尝试降级
WARNING llmops.retry: 主路径失败(TimeoutError: 主服务超时)，尝试降级
ERROR llmops.retry: 降级路径也失败，异常继续上抛


In [54]:
# 模板 5：事务上下文管理器（成功提交 / 失败回滚 / 异常不吞）
from contextlib import contextmanager

@contextmanager
def db_transaction(session):
    session["status"] = "begin"
    try:
        yield session
    except Exception:
        session["status"] = "rollback"
        raise                            # 关键：回滚后重新抛出
    else:
        session["status"] = "commit"

session = {}
with db_transaction(session) as s:
    s["rows"] = 1
print("成功事务:", session)

session = {}
try:
    with db_transaction(session) as s:
        raise RuntimeError("写入失败")
except RuntimeError as e:
    print("失败事务:", session, "| 异常未吞:", e)

成功事务: {'status': 'commit', 'rows': 1}
失败事务: {'status': 'rollback'} | 异常未吞: 写入失败


# 内置 / 生态对照

> **本节解决什么问题**：标准库已经提供了哪些异常处理能力？什么时候该引入第三方库？

## 标准库能力

| 能力 | 用法 | 一句话定位 |
|------|------|------------|
| `try/except/else/finally` | 语言内置 | 四段式结构化处理 |
| `raise ... from ...` | 语言内置 | 显式异常链（保留根因） |
| `ExceptionGroup` / `except*` | 3.11+ 内置 | 多异常一次抛出 |
| `contextlib.suppress` | 标准库 | 声明式吞掉预期内异常 |
| `contextlib.contextmanager` | 标准库 | 用生成器写 `__enter__/__exit__` |
| `logging.exception` | 标准库 | error 级日志 + 自动 traceback |
| `traceback` 模块 | 标准库 | 程序化解析/格式化 traceback |
| `assert` | 语言内置 | 内部不变量（`-O` 下移除） |

## 常用第三方库

| 库 | 能力 | 何时选它 | 注意点 |
|----|------|----------|--------|
| `tenacity` | 声明式重试（`@retry` 装饰器） | 重试逻辑变复杂（条件、停止策略、钩子） | 别对不可重试异常用 |
| `backoff` | 指数退避重试 | 只需要退避，不想引大依赖 | 与 tenacity 二选一 |
| `exceptiongroup` | 3.11 前的 ExceptionGroup backport | 老版本需要异常组 | 新项目无需 |
| `returns` | Result/Either 模式 | 团队约定"不用异常表达失败" | 学习成本高 |
| `structlog` | 结构化日志 | 需要把异常结构化进日志链路 | 与 logging.handler 配合 |
| `sentry-sdk` | 异常聚合上报 | 线上错误追踪 | 注意 PII 脱敏 |

> 选型原则：**先用标准库表达清楚，复杂度证明必要时再引入第三方**。重试、退避、断路器这三件事，标准库都能写出生产级实现（见第 9 章与综合实战）。

# 综合实战：LLMClient —— 把全部知识点组装成一个可用的客户端

> **本节解决什么问题**：知识点都会了，怎么在真实代码里组合？以下为一个面向 `llmops-api` 的 LLM 调用客户端，覆盖：异常层次、底层转译、限流重试、批量聚合、缓存降级、用户友好报错。

**需求清单**

1. 统一异常层次：调用方一个 `except LLMServiceError` 兜住全部
2. 底层错误转译：HTTP 状态码 → 业务异常
3. 限流自动重试；配额不足**不**重试（快速上抛）
4. 重试耗尽 → 降级缓存（留痕）
5. 批量调用：失败聚合成 `ExceptionGroup`，一次看全

In [55]:
# 综合实战（1/4）：异常层次 —— 对外一套语言，对内各归其位
class LLMServiceError(Exception):
    """llmops-api 所有 LLM 相关异常的基类"""

class TransportError2(LLMServiceError):
    """网络/网关层错误（可重试）"""
    def __init__(self, message, endpoint=None):
        self.endpoint = endpoint
        super().__init__(message)

class RateLimitError2(LLMServiceError):
    """限流（可重试），携带服务端建议的等待秒数"""
    def __init__(self, retry_after=1):
        self.retry_after = retry_after
        super().__init__(f"限流，建议 {retry_after}s 后重试")

class InsufficientQuotaError(LLMServiceError):
    """配额不足：重试无意义，应直接告知用户"""

print("层次自检:",
      issubclass(TransportError2, LLMServiceError),
      issubclass(RateLimitError2, LLMServiceError),
      issubclass(InsufficientQuotaError, LLMServiceError))

层次自检: True True True


In [56]:
# 综合实战（2/4）：底层转译 —— 网关响应 → 业务异常
class _FakeGateway:
    """确定性模拟网关：按 prompt 关键词决定响应，输出可复现"""
    def __init__(self):
        self.limit_hits = 0

    def post(self, endpoint, payload):
        prompt = payload.get("prompt", "")
        if "LIMIT" in prompt:
            if self.limit_hits < 1:            # 第 1 次限流，之后放行（演示重试成功）
                self.limit_hits += 1
                return {"status": 429, "body": {"retry_after": 2}}
            return {"status": 200, "body": {"content": "回答（重试后成功）"}}
        if "QUOTA" in prompt:
            return {"status": 402, "body": {"message": "配额不足"}}
        if "BOOM" in prompt:
            return {"status": 500, "body": {"error": "upstream crashed"}}
        return {"status": 200, "body": {"content": f"回答: {prompt}"}}

def _request(gateway, payload, endpoint="/v1/chat/completions"):
    """单次调用：把 HTTP 状态码翻译成异常层次中的具体类型"""
    resp = gateway.post(endpoint, payload)
    status, body = resp["status"], resp["body"]
    if status == 200:
        return body["content"]
    if status == 429:
        raise RateLimitError2(body.get("retry_after", 1))
    if status == 402:
        raise InsufficientQuotaError(body.get("message", "配额不足"))
    raise TransportError2(f"网关错误 {status}", endpoint)

gw = _FakeGateway()
for prompt in ("你好", "LIMIT 触发限流", "QUOTA 配额检查", "BOOM 崩溃"):
    try:
        print("→", _request(gw, {"prompt": prompt}))
    except LLMServiceError as e:
        extra = f"（retry_after={e.retry_after}）" if isinstance(e, RateLimitError2) else ""
        print(f"→ {type(e).__name__}: {e}{extra}")

→ 回答: 你好
→ RateLimitError2: 限流，建议 2s 后重试（retry_after=2）
→ InsufficientQuotaError: 配额不足
→ TransportError2: 网关错误 500


In [57]:
# 综合实战（3/4）：LLMClient —— 重试、降级、批量聚合
logger = logging.getLogger("llmops.client")

class LLMClient:
    def __init__(self, gateway, cache=None, max_retries=3):
        self.gateway = gateway
        self.cache = cache or {}
        self.max_retries = max_retries

    def chat(self, prompt, use_cache_fallback=True):
        payload = {"prompt": prompt}
        last_err = None
        for attempt in range(1, self.max_retries + 1):
            try:
                return _request(self.gateway, payload)
            except RateLimitError2 as e:
                # 限流：可重试（教学环境不真等待；真实项目 sleep e.retry_after）
                last_err = e
                print(f"    [重试 {attempt}/{self.max_retries}] {e}")
            except InsufficientQuotaError:
                raise                          # 配额不足：重试无意义，快速上抛
            except TransportError2 as e:
                last_err = e                   # 网络错误：可重试
                print(f"    [重试 {attempt}/{self.max_retries}] {e}")
        # 重试耗尽 → 降级缓存（可用性换一致性，必须留痕）
        if use_cache_fallback and prompt in self.cache:
            logger.warning("chat 降级到缓存: %r", prompt)
            return self.cache[prompt]
        raise last_err

    def batch(self, prompts):
        """批量调用：失败聚合为 ExceptionGroup，一次看全"""
        results, errors = {}, {}
        for i, p in enumerate(prompts):
            try:
                results[i] = self.chat(p, use_cache_fallback=False)
            except LLMServiceError as e:
                errors[i] = e
        if errors:
            raise ExceptionGroup("批量调用部分失败", list(errors.values()))
        return [results[i] for i in range(len(prompts))]

In [58]:
# 综合实战（4/4）：跑通全部场景
client = LLMClient(gw, cache={"BOOM 网关崩溃": "（缓存）旧的崩溃回答"})
gw.limit_hits = 0                 # 重置网关限流状态：让场景 2 真正走一次"限流→重试→成功"

# 场景 1：正常调用
print("场景 1（正常调用）:", client.chat("你好"))

# 场景 2：限流 → 自动重试 → 成功（下方重试行属于本场景）
print("场景 2（限流重试）:")
print("  结果:", client.chat("LIMIT 触发限流"))

# 场景 3：配额不足 → 不重试，快速上抛
print("场景 3（配额不足，不重试）:")
try:
    client.chat("QUOTA 配额不足")
except InsufficientQuotaError as e:
    print("  快速上抛 →", e)

# 场景 4：网关崩溃 → 重试耗尽 → 缓存降级（留痕）
print("场景 4（重试耗尽，降级缓存）:")
print("  结果:", client.chat("BOOM 网关崩溃"))
flush_log()

# 场景 5：批量调用 → 失败聚合为异常组
print("场景 5（批量聚合）:")
try:
    client.batch(["你好", "BOOM 甲", "QUOTA 乙"])
except ExceptionGroup as eg:
    print(" ", eg.message, "→", [type(e).__name__ for e in eg.exceptions])

场景 1（正常调用）: 回答: 你好
场景 2（限流重试）:
    [重试 1/3] 限流，建议 2s 后重试
  结果: 回答（重试后成功）
场景 3（配额不足，不重试）:
  快速上抛 → 配额不足
场景 4（重试耗尽，降级缓存）:
    [重试 1/3] 网关错误 500
    [重试 2/3] 网关错误 500
    [重试 3/3] 网关错误 500
  结果: （缓存）旧的崩溃回答
WARNING llmops.client: chat 降级到缓存: 'BOOM 网关崩溃'
场景 5（批量聚合）:
    [重试 1/3] 网关错误 500
    [重试 2/3] 网关错误 500
    [重试 3/3] 网关错误 500
  批量调用部分失败 → ['TransportError2', 'InsufficientQuotaError']


## 设计取舍说明

| 决策 | 理由 |
|------|------|
| 异常层次以 `LLMServiceError` 为根 | 调用方一个 except 就能兜住全部 LLM 相关错误，也可以精确捕获具体子类 |
| 配额不足**不**重试 | 重试设计第一步是分类「可重试 / 不可重试」：402 重试一万次也不会成功 |
| `RateLimitError` 携带 `retry_after` | 把"重试策略"从调用方移进异常本身，服务端建议直达重试逻辑 |
| `batch` 用 `ExceptionGroup` | 批处理关心的是"全部失败了什么"，单个异常装不下 |
| 降级放在重试耗尽之后、且可关闭 | 降级 = 用一致性换可用性，缓存旧答案必须 `logger.warning` 留痕 |
| 对外报错用固定话术 | 内部细节（traceback、endpoint）只进日志，绝不透给终端用户 |
| 异常链：对内 `from e`，对外 `from None` | 排查要根因，展示要安全——同一机制两种用法 |

# 常见陷阱：症状 / 解法速查表

> **本节解决什么问题**：把散落各章的坑汇成一张表——每行都指向可运行的复现代码。

| # | 陷阱 | 症状 | 解法 | 复现位置 |
|---|------|------|------|----------|
| 1 | 裸 `except` | Ctrl+C / sys.exit() 失效 | `except Exception:` 起步，尽量具体 | 知识点 10 |
| 2 | 静默吞异常 | 拿到 None，问题无从查起 | 记日志 + `raise` | 反例 1 |
| 3 | 过宽捕获 | 误把别人的 bug 当自己的处理 | 捕获具体类型 | 知识点 5/9 |
| 4 | `finally` 中 `return` | 返回值被静默覆盖 | finally 只清理 | 知识点 6 |
| 5 | `finally` 中抛异常 | 原异常丢失 | 同上 | 知识点 7 |
| 6 | 忘记 `raise` | 异常被吞，调用者不知道 | 处理不了就重新抛 | 反例 1 |
| 7 | 异常链丢失 | 因果关系不明 | `raise ... from e` | 下方复现单元 |
| 8 | `raise e` 代替裸 `raise` | traceback 多一层重复帧 | 直接 `raise` | 反例 3（含修正说明） |
| 9 | 用异常控制流程 | 高频路径变慢、难读 | `.get()` / `in` | 反例 4 |
| 10 | 异常开销大 | 正常路径也抛异常 | 可预期失败用返回值 | 知识点 37 |
| 11 | 忘 `super().__init__` | `str(e)` 为空 | 调用 super | 反例 2 |
| 12 | 异常组兼容性 | 3.10- 上 `except*` 语法错误 | 版本判断 + 聚合异常降级 | 下方复现单元 |
| 13 | `assert` 被优化 | `python -O` 后校验消失 | 参数校验用 if + raise | 知识点 31 |
| 14 | 循环中掩盖错误 | 批量任务失败数不明 | 收集后统一汇报 | 反例 5 |

In [59]:
# 陷阱复现：异常链丢失 —— 不写 from，因果"语义"就丢了
def no_from():
    try:
        {}["k"]
    except KeyError:
        raise ValueError("缺少配置项")            # ← 没写 from：只是隐式 __context__

def with_from():
    try:
        {}["k"]
    except KeyError as e:
        raise ValueError("缺少配置项") from e     # ← 显式 __cause__

for fn in (no_from, with_from):
    try:
        fn()
    except ValueError as e:
        cause = type(e.__cause__).__name__ if e.__cause__ else "None"
        print(f"{fn.__name__:9} → __cause__ = {cause}")
# 两者都自动记录 __context__，但只有 with_from 有明确的 __cause__
# （traceback 中显示为 "direct cause"，而不是含混的 "During handling"）

no_from   → __cause__ = None
with_from → __cause__ = KeyError


In [60]:
# 陷阱复现：异常组兼容性 —— 3.10 及以下没有 except*
import sys

print("当前版本:", sys.version_info[:3], "→ 支持 except* :", sys.version_info >= (3, 11))

# 3.10- 的等价降级方案：自定义聚合异常（结构见知识点 34 的 ValidationErrors）
class AggregatedErrors(Exception):
    def __init__(self, errors):
        self.errors = errors
        super().__init__(f"共 {len(errors)} 个错误: " + "; ".join(map(str, errors)))

try:
    raise AggregatedErrors([ValueError("a"), ValueError("b")])
except AggregatedErrors as e:
    print("降级方案:", e)

当前版本: (3, 13, 12) → 支持 except* : True
降级方案: 共 2 个错误: a; b


# 总结

## 通用代码模板（可直接复制）

```python
# ── 1. 完整语法骨架 ──────────────────────────────
try:
    result = risky()
except SpecificError as e:          # ① 具体异常在前
    handle(e)
except (ErrorA, ErrorB) as e:       # ② 元组：多类型
    handle(e)
except Exception as e:              # ③ 兜底放最后（不要裸 except）
    logger.exception("处理失败")     #    自动带 traceback
    raise                           # ④ 处理不了就重新抛
else:
    process(result)                 # ⑤ 无异常才执行：缩小 try 范围
finally:
    cleanup()                       # ⑥ 只清理，不 return 不 raise

# ── 2. 异常链 ────────────────────────────────────
raise BusinessError("对外消息") from original   # 保留根因
raise BusinessError("对外消息") from None       # 对外隐藏内部细节

# ── 3. 异常组（3.11+）────────────────────────────
raise ExceptionGroup("多个错误", [e1, e2, e3])
try:
    ...
except* ValueError as eg:           # 按类型槽分捕
    for e in eg.exceptions:
        ...

# ── 4. 自定义异常层次 ────────────────────────────
class AppError(Exception):
    """项目异常根：继承 Exception，绝不继承 BaseException"""

class ValidationError(AppError):
    def __init__(self, field, message):
        self.field = field
        self.message = message
        super().__init__(f"{field}: {message}")   # 忘了 super，str(e) 就空了
```

## 记忆口诀

1. **捕获要具体**，裸 `except` 是大忌
2. **吞异常必留痕**，处理不了就 `raise`
3. **转译要 `from`**，根因不能丢
4. **finally 只善后**，别 return 别抛错
5. **校验攒一起**，`ExceptionGroup` 报全集
6. **可预期用返回值，意外才用异常**

## 自测练习（先自己写，再看参考实现）

**练习 1**：实现 `safe_divide(a, b)`，返回 `Ok(结果)` 或 `Err(原因)`，不抛任何异常。

**练习 2**：实现装饰器 `@retry_on(times, exceptions)`——前 N-1 次失败最后成功则返回结果；始终失败则抛出最后一次的异常。

**练习 3**：实现 `validate_user(data)`，把所有字段错误攒进 `ExceptionGroup`（内含自定义异常），并能被 `except*` 分别捕获。

In [61]:
# 练习 1 参考实现：Result 风格的 safe_divide
from dataclasses import dataclass
from typing import Any

@dataclass
class Ok2:
    value: Any

@dataclass
class Err2:
    error: str

def safe_divide(a, b):
    if b == 0:
        return Err2("除数不能为 0")        # 可预期失败：用返回值表达
    return Ok2(a / b)

# 自测
assert isinstance(safe_divide(6, 3), Ok2) and safe_divide(6, 3).value == 2.0
assert isinstance(safe_divide(1, 0), Err2)
print("练习 1 通过：", safe_divide(6, 3), "|", safe_divide(1, 0))

练习 1 通过： Ok2(value=2.0) | Err2(error='除数不能为 0')


In [62]:
# 练习 2 参考实现：可配置的重试装饰器
from functools import wraps

def retry_on(times=3, exceptions=(Exception,)):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            for attempt in range(1, times + 1):
                try:
                    return func(*args, **kwargs)
                except exceptions:
                    if attempt == times:
                        raise              # 最后一次失败：原样抛出
            return None
        return wrapper
    return decorator

# 自测 1：前 2 次失败、第 3 次成功
_calls = {"n": 0}

@retry_on(times=3, exceptions=(ConnectionError,))
def flaky():
    _calls["n"] += 1
    if _calls["n"] < 3:
        raise ConnectionError("抖动")
    return "ok"

assert flaky() == "ok" and _calls["n"] == 3

# 自测 2：始终失败 → 抛最后一次的异常
@retry_on(times=2, exceptions=(ConnectionError,))
def always():
    raise ConnectionError("永久故障")

try:
    always()
    raise AssertionError("不应该走到这里")
except ConnectionError as e:
    assert str(e) == "永久故障"

print("练习 2 通过：成功路径与最终失败路径都符合预期")

练习 2 通过：成功路径与最终失败路径都符合预期


In [63]:
# 练习 3 参考实现：攒错校验 + 异常组 + except* 分捕
class FieldError(Exception):
    """字段错误基类"""

class NameMissing(FieldError):
    pass

class TypeInvalid(FieldError):
    pass

def validate_user(data):
    errors = []
    if not data.get("name"):
        errors.append(NameMissing("name 不能为空"))
    if not isinstance(data.get("age"), int):
        errors.append(TypeInvalid("age 必须是 int"))
    if errors:
        raise ExceptionGroup("用户数据校验失败", errors)

try:
    try:
        validate_user({"name": "", "age": "18"})
    except* NameMissing as eg:                      # 只接住 NameMissing 槽
        print("捕获 NameMissing:", [str(e) for e in eg.exceptions])
except ExceptionGroup as eg:                        # 剩余的继续向外传播
    print("剩余未处理:", [type(e).__name__ for e in eg.exceptions])
print("练习 3 通过")

捕获 NameMissing: ['name 不能为空']
剩余未处理: ['TypeInvalid']
练习 3 通过


## 结语

> **异常不是错误，而是控制流的一部分。好的异常设计让调用者知道「出了什么问题、为什么、该怎么办」，而不是只看到一个 traceback。**

（完）